In [ ]:
library(Seurat)
library(ggplot2)
library(MetaNeighbor)
library(SingleCellExperiment)
library(dplyr)

In [ ]:
library(tidyverse)
library(scales)
#library(circlize)
#library(ComplexHeatmap)
#library(pvclust)
library(stringr)

In [ ]:
hm <- readRDS('data/processed/excitatory_neuron_file')
DefaultAssay(hm) <- 'RNA'
hm@meta.data$species <- 'human'
hm

In [ ]:
sb <- readRDS('data/processed/excitatory_neuron_file')
DefaultAssay(sb) <- 'RNA'
sb@meta.data$species <- 'SB'
sb

In [ ]:
na <- readRDS('data/processed/excitatory_neuron_file')
DefaultAssay(na) <- 'RNA'
na@meta.data$species <- 'NA'
na

In [ ]:
sa <- readRDS('data/processed/excitatory_neuron_file')
DefaultAssay(sa) <- 'RNA'
sa@meta.data$species <- 'SA'
sa

In [ ]:
mu <- readRDS('data/processed/excitatory_neuron_file')
#mu <- readRDS('data/processed/cortex_analysis_file')
DefaultAssay(mu) <- 'RNA'
mu@meta.data$species <- 'mouse'
mu

In [ ]:
#mu <- readRDS('data/processed/excitatory_neuron_file')
mu <- readRDS('data/processed/cortex_analysis_file')
DefaultAssay(mu) <- 'RNA'
mu@meta.data$species <- 'mouse'
mu

In [ ]:
Idents(hm) <- hm@meta.data$cellType_layer
table(hm@meta.data$cellType_layer)

In [ ]:
Idents(sb) <- sb@meta.data$sub2
table(sb@meta.data$sub2)

In [ ]:
Idents(na) <- na@meta.data$annotation
table(na@meta.data$annotation)

In [ ]:
Idents(sa) <- sa@meta.data$annotation
table(sa@meta.data$annotation)

In [ ]:
#Idents(mu) <- mu@meta.data$ctx_layer
#table(mu@meta.data$ctx_layer)
Idents(mu) <- mu@meta.data$Layer
table(mu@meta.data$Layer)

In [ ]:
sa <- subset(sa, subset = annotation != "Other")
na <- subset(na, subset = annotation != "Other")

In [ ]:
#saveRDS(sa, file = "data/processed/excitatory_neuron_file")
#saveRDS(na, file = "data/processed/excitatory_neuron_file")

In [ ]:
# 提取表达矩阵和元数据
exprs_hm <- GetAssayData(hm)
exprs_sb <- GetAssayData(sb)
exprs_na <- GetAssayData(na)
exprs_sa <- GetAssayData(sa)
exprs_mu <- GetAssayData(mu)

meta_hm <- hm@meta.data
meta_sb <- sb@meta.data
meta_na <- na@meta.data
meta_sa <- sa@meta.data
meta_mu <- mu@meta.data

# 提取共有基因
common_genes <- Reduce(intersect, list(rownames(exprs_hm),rownames(exprs_sb), rownames(exprs_na), rownames(exprs_mu), rownames(exprs_sa)))

# 提取共有基因表达矩阵
exprs_hm_common <- exprs_hm[common_genes, ]
exprs_sb_common <- exprs_sb[common_genes, ]
exprs_na_common <- exprs_na[common_genes, ]
exprs_sa_common <- exprs_sa[common_genes, ]
exprs_mu_common <- exprs_mu[common_genes, ]

# 合并表达矩阵
combined_exprs <- cbind(exprs_hm_common, exprs_sb_common, exprs_na_common, exprs_mu_common, exprs_sa_common)

In [ ]:
new_colData <- data.frame(
  study_id = c(rep('HM', ncol(exprs_hm_common)),rep('SB', ncol(exprs_sb_common)), rep('NA', ncol(exprs_na_common)), rep('MU', ncol(exprs_mu_common)), rep('SA', ncol(exprs_sa_common))),
  cell_type = c(Idents(hm), Idents(sb), Idents(na), Idents(mu), Idents(sa))
)

# 创建 SingleCellExperiment 对象
sce_combined <- SingleCellExperiment(assays = list(RNA = combined_exprs), colData = new_colData)

# 检查合并后的对象
sce_combined

In [ ]:
table(sce_combined$study_id)

In [ ]:
table(sce_combined$cell_type)

In [ ]:
var_genes <- variableGenes(dat = sce_combined, exp_labels = sce_combined$study_id)
var_genes

In [ ]:
celltype_NV <- MetaNeighborUS(var_genes = var_genes,
                              dat = sce_combined,
                              study_id = sce_combined$study_id,
                              cell_type = sce_combined$cell_type,
                              fast_version = TRUE)
head(celltype_NV)

In [ ]:
png("heatmap_integrated_5_newmu.png",width = 1200, height = 1200)
#png("heatmap_integrated_5_pf.png",width = 1200, height = 1200)
plotHeatmapPretrained(aurocs = celltype_NV, cex = 1, margins = c(16, 16))
dev.off()

In [ ]:
#pdf("heatmap_integrated_5.pdf", width = 12, height = 12)
pdf("heatmap_integrated_5_newmu.pdf", width = 12, height = 12)
#pdf("heatmap_integrated_5_pf.pdf", width = 12, height = 12)
plotHeatmapPretrained(aurocs = celltype_NV, cex = 1, margins = c(16, 16))
dev.off()